# ARC-AGI-3 sg_kaggle_agent Submission (thin-shell pattern)

Mirrors `0.15`'s split: this notebook is a 3-cell thin shell that installs
official wheels and delegates everything else to `notebooks/sg_submit.py`
via `runpy.run_path`. All logic (path discovery, gateway wait, runner
setup, agent registration, run, dummy parquet) lives in `sg_submit.py`
so changes ship through dataset version bumps without touching notebook
cell content.

Logic split (mirrors `0.15/notebooks/arc_agi_3_kaggle_smoke.ipynb`):
- Cell 1: pip install `arc-agi` + `python-dotenv` from competition wheels.
- Cell 2: recursive-glob locate `sg_submit.py`, set `ARC_AGI_3_PRE_ROOT`
  env var, run script under `__main__` via `runpy.run_path`.
  `sg_submit.main()` decides Edit-mode (dummy parquet) vs RERUN
  (real submission) based on `KAGGLE_IS_COMPETITION_RERUN`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

WHEELS_DIR = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels')
if WHEELS_DIR.is_dir():
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        '--disable-pip-version-check', '--no-index', '--no-warn-conflicts',
        '--find-links', str(WHEELS_DIR),
        'arc-agi', 'python-dotenv',
    ], check=True)
else:
    print(f'Wheels not found at {WHEELS_DIR}; using preinstalled if available.')

import arc_agi
import arcengine
print('ARC packages imported:', arc_agi.__file__, arcengine.__file__)


In [ ]:
from pathlib import Path
import os
import runpy
import sys

input_root = Path(os.environ.setdefault('KAGGLE_INPUT_ROOT', '/kaggle/input'))
SCRIPT_NAME = 'sg_submit.py'
AGENT_MARKER = 'src/agents/sg_kaggle_agent.py'


def find_script_and_repo():
    """Find sg_submit.py paired with src/agents/sg_kaggle_agent.py.

    Scheme-agnostic: works for /kaggle/input/<slug>/ (legacy top-level mount)
    and /kaggle/input/datasets/<user>/<slug>/ (current per-user mount).
    """
    candidates = list(dict.fromkeys([
        Path('/kaggle/input/arc-agi-3-pre/notebooks') / SCRIPT_NAME,
        Path('/kaggle/input/arc-agi-3-pre-v1/notebooks') / SCRIPT_NAME,
        *input_root.glob(f'**/notebooks/{SCRIPT_NAME}'),
    ]))
    for cand in candidates:
        if not cand.exists():
            continue
        for ancestor in [cand.parent, *cand.parents]:
            if (ancestor / AGENT_MARKER).exists():
                return cand, ancestor
    return None, None


script_path, repo_root = find_script_and_repo()
if script_path is None:
    print('Mounted /kaggle/input entries:')
    if input_root.exists():
        for child in sorted(input_root.iterdir()):
            print(' -', child)
    raise FileNotFoundError(
        f'Could not find {SCRIPT_NAME} paired with {AGENT_MARKER} under '
        f'/kaggle/input. Attach the arc-agi-3-pre dataset to this notebook.'
    )

os.environ['ARC_AGI_3_PRE_ROOT'] = str(repo_root)
print('Using repo root:', repo_root)
print('Using submission script:', script_path)


def run_script(*args):
    sys.argv = [str(script_path), *args]
    runpy.run_path(str(script_path), run_name='__main__')


run_script('--dry-run')
print('Submission import dry-run passed')
run_script()
